In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import GCNConv

In [3]:
GRAPH_PATH = "../data/processed/graph_data.pt"

graph_data = torch.load(
    GRAPH_PATH,
    weights_only=False
)

print(graph_data)

Data(x=[4404, 136], edge_index=[2, 50000], y=[4404], train_mask=[4404], val_mask=[4404], test_mask=[4404])


In [4]:
device = torch.device("cpu")

graph_data = graph_data.to(device)

print("Using device:", device)

Using device: cpu


In [5]:
class GCNModel(nn.Module):

    def __init__(self, input_features, hidden_features, output_classes):
        super().__init__()

        self.conv1 = GCNConv(
            input_features,
            hidden_features
        )

        self.conv2 = GCNConv(
            hidden_features,
            hidden_features
        )

        self.classifier = nn.Linear(
            hidden_features,
            output_classes
        )

    def forward(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = F.dropout(
            x,
            p=0.5,
            training=self.training
        )

        x = self.conv2(x, edge_index)
        x = F.relu(x)

        x = self.classifier(x)

        return x

In [6]:
input_features = graph_data.num_node_features
hidden_features = 32
output_classes = 2

model = GCNModel(
    input_features=input_features,
    hidden_features=hidden_features,
    output_classes=output_classes
).to(device)

print(model)

GCNModel(
  (conv1): GCNConv(136, 32)
  (conv2): GCNConv(32, 32)
  (classifier): Linear(in_features=32, out_features=2, bias=True)
)


In [7]:
model.eval()

with torch.no_grad():

    output = model(
        graph_data.x,
        graph_data.edge_index
    )

print("Output shape:", output.shape)
print("Expected shape:", (graph_data.num_nodes, 2))

Output shape: torch.Size([4404, 2])
Expected shape: (4404, 2)


In [8]:
predicted_classes = output.argmax(dim=1)

print("First 20 predictions:")
print(predicted_classes[:20])

print("\nFirst 20 actual labels:")
print(graph_data.y[:20])

First 20 predictions:
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

First 20 actual labels:
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])


In [9]:
from sklearn.metrics import accuracy_score

initial_accuracy = accuracy_score(
    graph_data.y.cpu().numpy(),
    predicted_classes.cpu().numpy()
)

print("Initial accuracy:", initial_accuracy)

Initial accuracy: 0.9813805631244323


In [10]:
MODEL_PATH = "../data/processed/gcn_model_structure.pt"

torch.save(
    model.state_dict(),
    MODEL_PATH
)

print("Model structure saved successfully.")

Model structure saved successfully.


In [11]:
assert output.shape == (
    graph_data.num_nodes,
    output_classes
)

print("DAY 6 COMPLETED SUCCESSFULLY")
print("GCN model is ready for training.")

DAY 6 COMPLETED SUCCESSFULLY
GCN model is ready for training.
